In [1]:
pip install torch

In [1]:
!pip install pin
import pinocchio
print(dir(pinocchio))

['ACCELERATION', 'ADMMContactSolver', 'ARG0', 'ARG1', 'ARG2', 'ARG3', 'ARG4', 'AngleAxis', 'ArgumentPosition', 'BODY', 'BaumgarteCorrectorParameters', 'BroadPhaseManager_DynamicAABBTreeArrayCollisionManager', 'BroadPhaseManager_DynamicAABBTreeCollisionManager', 'BroadPhaseManager_IntervalTreeCollisionManager', 'BroadPhaseManager_NaiveCollisionManager', 'BroadPhaseManager_SSaPCollisionManager', 'BroadPhaseManager_SaPCollisionManager', 'COLLISION', 'CachedMeshLoader', 'CollisionCallBackBase', 'CollisionCallBackDefault', 'CollisionGeometry', 'CollisionObject', 'CollisionPair', 'CollisionResult', 'ComputeCollision', 'ComputeDistance', 'Contact', 'ContactCholeskyDecomposition', 'ContactType', 'Convention', 'CoulombFrictionCone', 'Data', 'DelassusCholeskyExpression', 'DelassusOperatorDense', 'DelassusOperatorSparse', 'DistanceResult', 'DualCoulombFrictionCone', 'Exception', 'FIXED_JOINT', 'Force', 'Frame', 'FrameType', 'GeometryData', 'GeometryModel', 'GeometryNoMaterial', 'GeometryObject', 

In [2]:
pip install numpy scipy

In [6]:
import pinocchio as pin
import torch
import time

# ----------------------------
# GPU + dtype setup
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64  # match numpy double precision
print("Using device:", device)

# ----------------------------
# Algorithm Implementations
# ----------------------------
def gauss_jordan(M, b):
    return torch.linalg.solve(M, b)

def neumann_series_inverse(M, num_terms=10):
    M0 = torch.diag(torch.diag(M))
    M0_inv = torch.linalg.inv(M0)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - M0_inv @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_terms):
        term = term @ E
        S = S + term
    return S @ M0_inv

def spai_inverse(M):
    return torch.diag(1.0 / torch.diag(M))

def hala(M, b, num_neumann=5):
    G_spai = spai_inverse(M)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - G_spai @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    qddot_hala = M_inv_hala @ b
    error = torch.linalg.norm(M @ qddot_hala - b)
    if error > 1e-3:
        qddot_hala = gauss_jordan(M, b)
    return qddot_hala

# ----------------------------
# Setup Pinocchio model
# ----------------------------
urdf_path = '/content/drive/MyDrive/ur5robot.urdf'
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq, nv = model.nq, model.nv

q_lower, q_upper = model.lowerPositionLimit, model.upperPositionLimit
qd_limit = model.velocityLimit

# Ensure finite, reasonable velocity limits
qd_limit_safe = qd_limit.copy()
qd_limit_safe[~torch.isfinite(torch.tensor(qd_limit_safe))] = 1.0
qd_limit_safe = torch.clamp(torch.tensor(qd_limit_safe), 0, 10).numpy()

# ----------------------------
# Random Sampling
# ----------------------------
num_runs = 1000
q_lower_t = torch.tensor(q_lower, dtype=dtype)
q_upper_t = torch.tensor(q_upper, dtype=dtype)
qd_limit_safe_t = torch.tensor(qd_limit_safe, dtype=dtype)

# Uniformly sample between lower and upper joint limits
q_samples = q_lower_t + (q_upper_t - q_lower_t) * torch.rand(num_runs, nq, dtype=dtype)
qd_samples = -qd_limit_safe_t + 2 * qd_limit_safe_t * torch.rand(num_runs, nv, dtype=dtype)

# ----------------------------
# Benchmark Setup
# ----------------------------
results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'neumann_error': [], 'spai_error': [], 'hala_error': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

# ----------------------------
# Benchmark Loop
# ----------------------------
for i in range(num_runs):
    q = q_samples[i].cpu().numpy()
    qd = qd_samples[i].cpu().numpy()
    pin.computeAllTerms(model, data, q, qd)

    # Convert Pinocchio outputs to GPU tensors
    M = torch.tensor(data.M, dtype=dtype, device=device)
    Cqd = torch.tensor(data.nle - data.g, dtype=dtype, device=device)
    g_vec = torch.tensor(data.g, dtype=dtype, device=device)
    tau = torch.ones(nv, dtype=dtype, device=device)
    b = tau - Cqd - g_vec

    # Perturbation for stability
    delta_M = torch.randn_like(M) * 1e-6
    M_pert = M + delta_M

    # --- Reference (Gauss-Jordan) ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_ref = gauss_jordan(M, b)
    torch.cuda.synchronize()
    t1 = time.time()
    results['ref_time'].append((t1 - t0) * 1000)

    qddot_ref_pert = gauss_jordan(M_pert, b)
    kappa_ref = (torch.linalg.norm(qddot_ref_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['ref_kappa'].append(kappa_ref.item())

    # --- Neumann Series ---
    torch.cuda.synchronize()
    t0 = time.time()
    neumann_inv = neumann_series_inverse(M)
    qddot_neumann = neumann_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['neumann_time'].append((t1 - t0) * 1000)
    results['neumann_error'].append((torch.linalg.norm(qddot_neumann - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_neumann_pert = neumann_series_inverse(M_pert) @ b
    kappa_neumann = (torch.linalg.norm(qddot_neumann_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                    (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['neumann_kappa'].append(kappa_neumann.item())

    # --- SPAI ---
    torch.cuda.synchronize()
    t0 = time.time()
    spai_inv = spai_inverse(M)
    qddot_spai = spai_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['spai_time'].append((t1 - t0) * 1000)
    results['spai_error'].append((torch.linalg.norm(qddot_spai - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    spai_inv_pert = spai_inverse(M_pert)
    qddot_spai_pert = spai_inv_pert @ b
    kappa_spai = (torch.linalg.norm(qddot_spai_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['spai_kappa'].append(kappa_spai.item())

    # --- HALA ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_hala = hala(M, b, num_neumann=10)
    torch.cuda.synchronize()
    t1 = time.time()
    results['hala_time'].append((t1 - t0) * 1000)
    results['hala_error'].append((torch.linalg.norm(qddot_hala - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_hala_pert = hala(M_pert, b, num_neumann=10)
    kappa_hala = (torch.linalg.norm(qddot_hala_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['hala_kappa'].append(kappa_hala.item())

# ----------------------------
# Summary
# ----------------------------
def summarize(name, times, errors, kappas):
    print(f"{name}: Avg Time (ms): {torch.tensor(times).mean():.2f}, "
          f"Avg Error: {torch.tensor(errors, dtype=torch.float64).mean():.2e}, "
          f"Avg Stability (kappa): {torch.tensor(kappas).mean():.2f}")

# FIXED LINE BELOW
summarize("Gauss-Jordan (ref)", results['ref_time'], [0.0]*num_runs, results['ref_kappa'])
summarize("Neumann Series", results['neumann_time'], results['neumann_error'], results['neumann_kappa'])
summarize("SPAI", results['spai_time'], results['spai_error'], results['spai_kappa'])
summarize("HALA", results['hala_time'], results['hala_error'], results['hala_kappa'])


Using device: cuda
Gauss-Jordan (ref): Avg Time (ms): 0.25, Avg Error: 0.00e+00, Avg Stability (kappa): 28.39
Neumann Series: Avg Time (ms): 0.84, Avg Error: 4.58e-01, Avg Stability (kappa): 335707.69
SPAI: Avg Time (ms): 0.18, Avg Error: 4.15e-01, Avg Stability (kappa): 250711.16
HALA: Avg Time (ms): 0.96, Avg Error: 0.00e+00, Avg Stability (kappa): 28.39


In [8]:
import pinocchio as pin
import torch
import time

# ----------------------------
# GPU + dtype setup
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64  # match numpy double precision
print("Using device:", device)

# ----------------------------
# Algorithm Implementations
# ----------------------------
def gauss_jordan(M, b):
    return torch.linalg.solve(M, b)

def neumann_series_inverse(M, num_terms=10):
    M0 = torch.diag(torch.diag(M))
    M0_inv = torch.linalg.inv(M0)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - M0_inv @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_terms):
        term = term @ E
        S = S + term
    return S @ M0_inv

def spai_inverse(M):
    return torch.diag(1.0 / torch.diag(M))

def hala(M, b, num_neumann=5):
    G_spai = spai_inverse(M)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - G_spai @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    qddot_hala = M_inv_hala @ b
    error = torch.linalg.norm(M @ qddot_hala - b)
    if error > 1e-3:
        qddot_hala = gauss_jordan(M, b)
    return qddot_hala

# ----------------------------
# Setup Pinocchio model
# ----------------------------
urdf_path = '/content/drive/MyDrive/robot_arm_2link.urdf'
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq, nv = model.nq, model.nv

q_lower, q_upper = model.lowerPositionLimit, model.upperPositionLimit
qd_limit = model.velocityLimit

# Ensure finite, reasonable velocity limits
qd_limit_safe = qd_limit.copy()
qd_limit_safe[~torch.isfinite(torch.tensor(qd_limit_safe))] = 1.0
qd_limit_safe = torch.clamp(torch.tensor(qd_limit_safe), 0, 10).numpy()

# ----------------------------
# Random Sampling
# ----------------------------
num_runs = 1000
q_lower_t = torch.tensor(q_lower, dtype=dtype)
q_upper_t = torch.tensor(q_upper, dtype=dtype)
qd_limit_safe_t = torch.tensor(qd_limit_safe, dtype=dtype)

# Uniformly sample between lower and upper joint limits
q_samples = q_lower_t + (q_upper_t - q_lower_t) * torch.rand(num_runs, nq, dtype=dtype)
qd_samples = -qd_limit_safe_t + 2 * qd_limit_safe_t * torch.rand(num_runs, nv, dtype=dtype)

# ----------------------------
# Benchmark Setup
# ----------------------------
results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'neumann_error': [], 'spai_error': [], 'hala_error': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

# ----------------------------
# Benchmark Loop
# ----------------------------
for i in range(num_runs):
    q = q_samples[i].cpu().numpy()
    qd = qd_samples[i].cpu().numpy()
    pin.computeAllTerms(model, data, q, qd)

    # Convert Pinocchio outputs to GPU tensors
    M = torch.tensor(data.M, dtype=dtype, device=device)
    Cqd = torch.tensor(data.nle - data.g, dtype=dtype, device=device)
    g_vec = torch.tensor(data.g, dtype=dtype, device=device)
    tau = torch.ones(nv, dtype=dtype, device=device)
    b = tau - Cqd - g_vec

    # Perturbation for stability
    delta_M = torch.randn_like(M) * 1e-6
    M_pert = M + delta_M

    # --- Reference (Gauss-Jordan) ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_ref = gauss_jordan(M, b)
    torch.cuda.synchronize()
    t1 = time.time()
    results['ref_time'].append((t1 - t0) * 1000)

    qddot_ref_pert = gauss_jordan(M_pert, b)
    kappa_ref = (torch.linalg.norm(qddot_ref_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['ref_kappa'].append(kappa_ref.item())

    # --- Neumann Series ---
    torch.cuda.synchronize()
    t0 = time.time()
    neumann_inv = neumann_series_inverse(M)
    qddot_neumann = neumann_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['neumann_time'].append((t1 - t0) * 1000)
    results['neumann_error'].append((torch.linalg.norm(qddot_neumann - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_neumann_pert = neumann_series_inverse(M_pert) @ b
    kappa_neumann = (torch.linalg.norm(qddot_neumann_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                    (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['neumann_kappa'].append(kappa_neumann.item())

    # --- SPAI ---
    torch.cuda.synchronize()
    t0 = time.time()
    spai_inv = spai_inverse(M)
    qddot_spai = spai_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['spai_time'].append((t1 - t0) * 1000)
    results['spai_error'].append((torch.linalg.norm(qddot_spai - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    spai_inv_pert = spai_inverse(M_pert)
    qddot_spai_pert = spai_inv_pert @ b
    kappa_spai = (torch.linalg.norm(qddot_spai_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['spai_kappa'].append(kappa_spai.item())

    # --- HALA ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_hala = hala(M, b, num_neumann=10)
    torch.cuda.synchronize()
    t1 = time.time()
    results['hala_time'].append((t1 - t0) * 1000)
    results['hala_error'].append((torch.linalg.norm(qddot_hala - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_hala_pert = hala(M_pert, b, num_neumann=10)
    kappa_hala = (torch.linalg.norm(qddot_hala_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['hala_kappa'].append(kappa_hala.item())

# ----------------------------
# Summary
# ----------------------------
def summarize(name, times, errors, kappas):
    print(f"{name}: Avg Time (ms): {torch.tensor(times).mean():.2f}, "
          f"Avg Error: {torch.tensor(errors, dtype=torch.float64).mean():.2e}, "
          f"Avg Stability (kappa): {torch.tensor(kappas).mean():.2f}")

# FIXED LINE BELOW
summarize("Gauss-Jordan (ref)", results['ref_time'], [0.0]*num_runs, results['ref_kappa'])
summarize("Neumann Series", results['neumann_time'], results['neumann_error'], results['neumann_kappa'])
summarize("SPAI", results['spai_time'], results['spai_error'], results['spai_kappa'])
summarize("HALA", results['hala_time'], results['hala_error'], results['hala_kappa'])


Using device: cuda
Gauss-Jordan (ref): Avg Time (ms): 0.20, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00
Neumann Series: Avg Time (ms): 0.63, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00
SPAI: Avg Time (ms): 0.14, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00
HALA: Avg Time (ms): 0.56, Avg Error: 0.00e+00, Avg Stability (kappa): 1.00


In [7]:
# make sure CUDA is installed
!nvcc --version

# make sure you have a GPU runtime (if this fails go to runtime -> change runtime type)
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Mon Nov 10 00:48:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0       